# 05 — Reliability-qualified NTL: 2012–2024 sensitivity

This notebook builds the exact revision sequence: (1) full-period regional raw direct, Gap-Filled, and Román-inspired four-day comparison with $r$, paired z-score RMSE, and retained $n$; (2) SC and GHSL qualification; (3) 1–30-calendar-day sensitivity curves and spatial-scale × window heatmaps at declared SC thresholds; (4) composite-weight sensitivity and held-out island-grid evaluation. Haiyan and Rai appear only as zoomed examples. **This notebook is unexecuted; run it on your RQ1 data and interpret the figures.**

## Data contract

The study clock is **19 January 2012–31 December 2024** throughout. The opening direct and Gap-Filled signals come from the **same Region VIII CSV you identified**, `Region VIII_NTL_VNP46A2.csv`, using `DNB_BRDF_Corrected_NTL` and `Gap_Filled_DNB_BRDF_Corrected_NTL`. The notebook searches the RQ1 project for this file; if it lives elsewhere, set `RQNTL_REGION_VIII_CSV` to its full path. Thus the raw–filled comparison has the same dates and exported regional footprint. The RQ1 direct regional `EasternVisayas_DNBBRDF_daily_stats.csv` supplies valid-pixel support and the 1–30-day sensitivity; eight GHSL masks and 1 AM NGCP load follow RQ1 Notebooks 03–04. No RQ2 data are loaded.

The four-day signal is a **non-overlapping mean of the companion CSV's direct regional daily series**, anchored to Haiyan as in the sample RQ2 notebook. RQ2 uses a pixel-first four-day mean and a recovery baseline; those cannot be exactly reconstructed from the RQ1 daily summary CSV. This is an explicitly labelled Román-inspired temporal analogue, not a pixel-first replication. Raw and Gap-Filled $n$ count paired **days**; four-day $n$ counts complete calendar **blocks**. All RMSE values use paired z-scores, so $\mathrm{RMSE}^2=2(1-r)$.


In [ ]:
from pathlib import Path
import os, re, math, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_columns = 100
ROOT = Path(os.environ.get('RQNTL_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'datasets'
def find_region_viii_csv():
    override = os.environ.get('RQNTL_REGION_VIII_CSV')
    if override:
        p = Path(override).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f'RQNTL_REGION_VIII_CSV does not exist: {p}')
        return p
    name = 'Region VIII_NTL_VNP46A2.csv'
    candidates = [DATA / 'vnp46a2/region8_daily_stats' / name,
                  DATA / 'vnp46a2' / name, DATA / name, ROOT / name]
    for p in candidates:
        if p.is_file(): return p
    for base in (DATA, ROOT, ROOT.parent):
        if not base.is_dir(): continue
        try:
            found = next(base.rglob(name), None)
        except OSError:
            found = None
        if found is not None: return found
    raise FileNotFoundError(f'{name} was not found under the RQ1 project. Set RQNTL_REGION_VIII_CSV to its full path.')

REGION_VIII_CSV = find_region_viii_csv()
REGIONAL_CSV = DATA / 'vnp46a2/region8_daily_stats/EasternVisayas_DNBBRDF_daily_stats.csv'
GHSL_DIR = DATA / 'vnp46a2/ghsl_masked_ntl_samar_leyte'
CROSS_DIR = DATA / 'vnp46a2/visayas_ghsl_masked_daily_stats'
NGCP_CSV = DATA / 'ngcp/NGCP_Hourly_Demand.csv'
NGCP_XLSX = DATA / 'ngcp/NGCP_hourly_load.xlsx'
OUT = Path(os.environ.get('RQNTL_OUTPUT_ROOT', ROOT / 'outputs/sensitivity/revision05')).expanduser().resolve()
OUT.mkdir(parents=True, exist_ok=True)

START, END = pd.Timestamp('2012-01-19'), pd.Timestamp('2024-12-31')
CALENDAR = pd.date_range(START, END, freq='D', name='date')
HAIYAN = pd.Timestamp('2013-11-08')
WINDOWS = list(range(1, 31))
TAUS = [0, 5, 10, 20, 30, 50, 70, 90]
TRANSFER_TAUS = [0, 5, 10, 20, 30, 50, 70]
MIN_WINDOW_FRACTION = 0.50  # only for the separately labelled window-supported scenario
PLATEAU_DELTA_R = 0.02       # sensitivity is repeated below for 0.01 and 0.05
MIN_DIRECT_COVERAGE = 0.20  # eligibility rule for transfer training islands
MIN_PAIRS = 5
GROUPS = {
    '30': ('G1', 'Urban Core', '#D65A4A'),
    '23-30': ('G2', 'Dense Urban', '#A87000'),
    '22-30': ('G3', 'Core & Inner', '#E6A23C'),
    '21-30': ('G4', 'Urban Fringe', '#D9B900'),
    '13-30': ('G5', 'Built-Up / Rural Mix', '#375623'),
    '12-30': ('G6', 'Low–Mid Rural', '#7BAF54'),
    '11-30': ('G7', 'Rural Settlements', '#A8CE74'),
    '10-30': ('G8', 'All Land GHSL classes', '#4E79A7'),
}
FULL_LABEL = 'Full RQ1 regional export'
MASK_ORDER = [FULL_LABEL] + [v[0] for v in GROUPS.values()]
COLOR = {FULL_LABEL: '#183A63', **{v[0]: v[2] for v in GROUPS.values()}}
print('RQ1 root:', ROOT)
print('Same-footprint Region VIII raw + Gap-Filled CSV:', REGION_VIII_CSV)
print('Long-record full-area input:', REGIONAL_CSV)
print('Output folder:', OUT)

def save_fig(fig, name):
    fig.show()
    fig.write_html(OUT / (name + '.html'), include_plotlyjs='cdn')
    try:
        fig.write_image(OUT / (name + '.png'), scale=2)
    except Exception as exc:
        print(f'{name}: HTML saved; PNG skipped ({type(exc).__name__}). Install kaleido for PNG.')

def date_from_frame(frame):
    d = frame.copy()
    if 'date' not in d:
        if 'system:index' not in d:
            raise ValueError('Expected date or system:index')
        d['date'] = d['system:index'].astype(str).str.extract(r'(\d{4}_\d{2}_\d{2})')[0].str.replace('_','-',regex=False)
    d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    if d['date'].isna().any() or d['date'].duplicated().any():
        raise ValueError('Missing or duplicate dates in an NTL input')
    return d.set_index('date').sort_index()

def ngcp_hour1(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    f = pd.read_csv(path, skiprows=2)
    if f.shape[1] < 4:
        raise ValueError('NGCP CSV layout differs from RQ1 Notebook 03')
    f.columns = ['Date','Coincident Peak','Hour No'] + [str(i) for i in range(1, f.shape[1]-2)]
    f['Date'] = pd.to_datetime(f['Date'], dayfirst=True, errors='coerce').dt.normalize()
    f['1'] = pd.to_numeric(f['1'], errors='coerce')
    f = f.dropna(subset=['Date']).set_index('Date').sort_index()
    if f.index.duplicated().any():
        raise ValueError('Duplicate NGCP dates')
    return f['1'].reindex(CALENDAR).rename('ngcp_1am_mw')

LOAD = ngcp_hour1(NGCP_CSV)

def paired_metrics(signal, target, eligible_index=None):
    x, y = pd.Series(signal, dtype=float), pd.Series(target, dtype=float)
    if eligible_index is None:
        eligible_index = y.index[y.notna()]
    eligible_index = pd.Index(eligible_index)
    p = pd.concat([x.rename('ntl'), y.rename('load')], axis=1).reindex(eligible_index).dropna()
    n, possible = len(p), len(eligible_index)
    result = {'n_pairs': n, 'n_possible': possible,
              'n_pct': 100*n/possible if possible else np.nan,
              'r': np.nan, 'rmse_z': np.nan}
    if n < MIN_PAIRS or p.ntl.std(ddof=0) == 0 or p.load.std(ddof=0) == 0:
        return result
    a = (p.ntl-p.ntl.mean()) / p.ntl.std(ddof=0)
    b = (p.load-p.load.mean()) / p.load.std(ddof=0)
    result['r'] = float(a.corr(b))
    result['rmse_z'] = float(np.sqrt(np.mean((a-b)**2)))
    return result

## 1. The three full-period signals before reliability qualification

**Purpose:** compare direct DNB-BRDF, its Gap-Filled companion, and a four-day direct composite for all of 2012–2024. The first two are columns from the **same Region VIII export**. The charts let us test whether direct radiance is erratic, whether apparent Gap-Filled continuity occurs when direct observations are sparse, and whether four-day aggregation improves load alignment. These are questions for the results, not conclusions assumed in advance.


In [ ]:
companion=date_from_frame(pd.read_csv(REGION_VIII_CSV))
required={'DNB_BRDF_Corrected_NTL','Gap_Filled_DNB_BRDF_Corrected_NTL'}
missing=required-set(companion.columns)
if missing:
    raise ValueError(f'{REGION_VIII_CSV.name}: missing {sorted(missing)}')
DIRECT_RAW=pd.to_numeric(companion['DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
GAP_RAW=pd.to_numeric(companion['Gap_Filled_DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
if DIRECT_RAW.notna().sum()<365 or GAP_RAW.notna().sum()<365:
    raise ValueError('The Region VIII CSV must contain the long RQ1 daily record, not an event-only subset.')
GAP_AVAILABLE=True
# Direct valid-pixel support comes from the RQ1 regional daily statistics export.
direct_frame=date_from_frame(pd.read_csv(REGIONAL_CSV))
if not {'NTL_mean','Valid_px'} <= set(direct_frame.columns):
    raise ValueError('RQ1 regional CSV must have NTL_mean and Valid_px')
DIRECT_PX=pd.to_numeric(direct_frame.Valid_px,errors='coerce').reindex(CALENDAR)
DIRECT_SC=100*DIRECT_PX/DIRECT_PX.max()

# Event-anchored, non-overlapping, complete four-calendar-day units.
block_id=np.floor_divide((CALENDAR-HAIYAN).days,4)
block_start=pd.DatetimeIndex(HAIYAN+pd.to_timedelta(np.unique(block_id)*4,unit='D'))
block_table=pd.DataFrame({'direct':DIRECT_RAW.to_numpy(),'gap':GAP_RAW.to_numpy(),
                          'load':LOAD.to_numpy(),'block':block_id},index=CALENDAR)
agg=block_table.groupby('block').agg(direct_mean=('direct','mean'),gap_mean=('gap','mean'),
                                      load_mean=('load','mean'),direct_days=('direct','count'),
                                      gap_days=('gap','count'),load_days=('load','count'),calendar_days=('load','size'))
agg.index=block_start
agg=agg[(agg.calendar_days==4)&(agg.load_days==4)]
agg['four_day_direct']=agg.direct_mean.where(agg.direct_days>0)
agg.to_csv(OUT/'opening_fullperiod_four_day_block_support.csv')
ELIGIBLE_DAYS=CALENDAR[LOAD.notna()]
ELIGIBLE_BLOCKS=agg.index
OPENING=pd.DataFrame([
 {'signal':'Direct DNB-BRDF','unit':'days',**paired_metrics(DIRECT_RAW,LOAD,ELIGIBLE_DAYS)},
 {'signal':'Gap-Filled companion','unit':'days',**paired_metrics(GAP_RAW,LOAD,ELIGIBLE_DAYS)},
 {'signal':'Román-inspired 4-day regional mean','unit':'4-day blocks',
  **paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
OPENING.to_csv(OUT/'opening_2012_2024_three_signal_metrics.csv',index=False)
print('Study calendar:',CALENDAR.min().date(),'to',CALENDAR.max().date())
print('Direct + Gap-Filled source:',REGION_VIII_CSV)
display(OPENING.round(3))

### Figure 1A — $r$, RMSE and retained $n$, 2012–2024

The first two denominators are load-available days; the third is complete four-day blocks with all four NGCP days. The metrics use their native clocks, so compare the broad pattern and availability rather than treating a daily $n$ as equivalent to a block $n$. For a common-clock diagnostic, the CSV below also aggregates raw and Gap-Filled daily regional values to the same four-day blocks.

In [ ]:
common=pd.DataFrame([
 {'signal':'Direct on common 4-day blocks',**paired_metrics(agg.direct_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
 {'signal':'Gap-Filled on common 4-day blocks',**paired_metrics(agg.gap_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
 {'signal':'Four-day direct regional composite',**paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
common.to_csv(OUT/'supplement_fullperiod_common_four_day_clock.csv',index=False)
if GAP_AVAILABLE:
    colors=['#4E79A7','#D65A4A','#375623']
    fig=make_subplots(rows=1,cols=3,subplot_titles=['Pearson r','Paired z-score RMSE','Retained analysis units (%)'])
    for col,field in enumerate(['r','rmse_z','n_pct'],1):
        vals=OPENING[field].to_numpy(float)
        labels=[f'{int(x.n_pairs)}/{int(x.n_possible)} {x.unit}' for x in OPENING.itertuples()]
        fig.add_trace(go.Bar(x=OPENING.signal,y=vals,marker_color=colors,showlegend=False,
                             text=labels if field=='n_pct' else [f'{v:.2f}' if np.isfinite(v) else 'undefined' for v in vals],
                             textposition='outside'),row=1,col=col)
    fig.update_yaxes(title_text='r',row=1,col=1)
    fig.update_yaxes(title_text='RMSE (paired SD units)',row=1,col=2)
    fig.update_yaxes(title_text='n / eligible (%)',range=[0,115],row=1,col=3)
    fig.update_xaxes(tickangle=-18)
    fig.update_layout(template='plotly_white',width=1370,height=500,
                      title='Full regional RQ1 signals, 2012–2024: alignment and availability',
                      margin=dict(l=65,r=30,t=85,b=145),font=dict(color='#233B60'))
    save_fig(fig,'fig_01a_fullperiod_three_signal_metrics')

### Figure 1B — The full period first; storm windows are zooms

The full 2012–2024 plot keeps the three products in context. The top strip is directly valid-pixel support from the raw band; high Gap-Filled availability is not new direct evidence. The four-day value is repeated only as a **step display** across its own block, never counted as four independent observations. Haiyan and Rai zooms use the same underlying series and are illustrative, not separate validation samples.

In [ ]:
def zview(s):
    s=pd.Series(s,dtype=float); sd=s.std(ddof=0)
    return (s-s.mean())/sd if np.isfinite(sd) and sd>0 else s*0+np.nan

def baseline_timeline(left,right,stem,title):
    ix=CALENDAR[(CALENDAR>=left)&(CALENDAR<=right)]
    fig=make_subplots(rows=4,cols=1,shared_xaxes=True,vertical_spacing=.045,
                      row_heights=[.12,.29,.29,.30],
                      subplot_titles=['Direct valid-pixel support','Direct DNB-BRDF','Gap-Filled companion','Four-day direct regional composite'])
    fig.add_trace(go.Scatter(x=ix,y=DIRECT_SC.reindex(ix),mode='lines',line=dict(color='#93AECB'),showlegend=False),row=1,col=1)
    for row,signal,color,name in [(2,DIRECT_RAW,'#4E79A7','Direct'),(3,GAP_RAW,'#D65A4A','Gap-Filled')]:
        fig.add_trace(go.Scatter(x=ix,y=zview(signal).reindex(ix),mode='lines',name=name,
                                 line=dict(color=color,width=1.5)),row=row,col=1)
        fig.add_trace(go.Scatter(x=ix,y=zview(LOAD).reindex(ix),mode='lines',name='NGCP 1 AM',
                                 showlegend=(row==2),line=dict(color='black',width=1.2)),row=row,col=1)
    blocks=agg[(agg.index<=right)&(agg.index+pd.Timedelta(days=3)>=left)]
    fig.add_trace(go.Scatter(x=blocks.index,y=zview(agg.four_day_direct).reindex(blocks.index),
                             mode='lines+markers',name='Four-day NTL',line=dict(color='#375623',shape='hv')),row=4,col=1)
    fig.add_trace(go.Scatter(x=blocks.index,y=zview(agg.load_mean).reindex(blocks.index),
                             mode='lines',name='Four-day NGCP',line=dict(color='black',shape='hv')),row=4,col=1)
    for event in [pd.Timestamp('2013-11-08'),pd.Timestamp('2021-12-16')]:
        if left<=event<=right: fig.add_vline(x=event,line_dash='dot',line_color='#233B60')
    fig.update_yaxes(title_text='SC %',range=[0,105],row=1,col=1)
    for row in [2,3,4]:fig.update_yaxes(title_text='z-score',row=row,col=1)
    fig.update_layout(template='plotly_white',width=1250,height=900,title=title,
                      legend=dict(orientation='h',y=-.09),margin=dict(l=75,r=30,t=90,b=95),font=dict(color='#233B60'))
    save_fig(fig,stem)
if GAP_AVAILABLE:
    baseline_timeline(START,END,'fig_01b_fullperiod_signal_timeline','Whole regional export, 2012–2024')
    baseline_timeline(pd.Timestamp('2013-09-01'),pd.Timestamp('2014-03-31'),
                      'fig_s_haiyan_three_signal_zoom','Haiyan zoom — same 2012–2024 inputs')
    baseline_timeline(pd.Timestamp('2021-10-01'),pd.Timestamp('2022-03-31'),
                      'fig_s_rai_three_signal_zoom','Rai zoom — same 2012–2024 inputs')

**Interpretation gate.** Inspect direct support alongside apparent Gap-Filled flat stretches; the CSV alone does not establish the exact gap-filling mechanism, so do not claim every flat section literally repeats the last observation. The opening direct and Gap-Filled columns are paired from one export. The SC panel comes from the RQ1 regional direct-statistics export and should be checked for footprint consistency before using it as a pixel-level explanation of the companion CSV.


## 2. Bring in the long RQ1 direct record and GHSL masks

**Question.** Does the relationship change when the spatial extent is narrowed from the full regional export to settlement-focused support? RQ1 Notebook 00 provides the regional direct-band daily CSV; Notebook 03 provides eight cumulative GHSL-mask CSVs. The former uses `NTL_mean`, while the latter's published analysis uses `Mean_by_sum`. Both are kept as their RQ1 sources, labelled in the inventory, and compared as standardized trajectories/metrics rather than raw radiance levels. A manuscript claim that the full regional export exactly matches the NGCP Samar–Leyte boundary requires a boundary audit.

For this qualified branch we follow Notebook 03's basic preprocessing: positive valid-pixel count, an upper 95th-percentile cap, and `SC = 100 × Valid_px / max(Valid_px)` within each input mask. The cap is applied **only after** the opening unqualified comparison. Here τ=0 retains all direct dates with valid pixels; higher τ rejects low-completeness dates.

In [ ]:
def read_daily_csv(path, value_col):
    if not path.is_file():
        raise FileNotFoundError(path)
    d = date_from_frame(pd.read_csv(path))
    missing = {value_col,'Valid_px'} - set(d.columns)
    if missing:
        raise ValueError(f'{path.name}: missing {sorted(missing)}')
    d['value'] = pd.to_numeric(d[value_col],errors='coerce')
    d['Valid_px'] = pd.to_numeric(d.Valid_px,errors='coerce')
    return d[['value','Valid_px']].reindex(CALENDAR)

def prep_direct(d):
    f=d.copy()
    valid=(f.Valid_px>0)&f.value.notna()&np.isfinite(f.value)
    if not valid.any():
        raise ValueError('No direct dates in a spatial scale')
    denominator=float(f.loc[valid,'Valid_px'].max())
    hi=float(f.loc[valid,'value'].quantile(.95))
    f['SC_pct']=100*f.Valid_px/denominator
    f['ntl']=f.value.clip(upper=hi).where(valid)
    return f[['ntl','Valid_px','SC_pct']],{'max_valid_px':denominator,'upper_95_cap':hi,'direct_dates':int(valid.sum())}

REGION_DAILY=read_daily_csv(REGIONAL_CSV,'NTL_mean')
SCALES={FULL_LABEL:prep_direct(REGION_DAILY)[0]}
inventory=[{'scale':FULL_LABEL,'source':str(REGIONAL_CSV),'value_col':'NTL_mean',
            **prep_direct(REGION_DAILY)[1]}]
for group,(mask,label,color) in GROUPS.items():
    path=GHSL_DIR / f'EasternVisayas_DNBBRDF_{group}_stats.csv'
    d=read_daily_csv(path,'Mean_by_sum')
    SCALES[mask],audit=prep_direct(d)
    inventory.append({'scale':mask,'source':str(path),'value_col':'Mean_by_sum',**audit})
INPUT_AUDIT=pd.DataFrame(inventory)
INPUT_AUDIT.to_csv(OUT/'qualified_input_inventory.csv',index=False)
print(f'Loaded long direct series for {len(SCALES)} spatial scales; source and denominator audit saved.')

### Figure 2 — Two reliability conditions, one at a time

The left diagnostic plots direct-date retention as the SC threshold rises for each spatial support. The right compares **unsmoothed, 95th-percentile-capped** $r$ at τ=0 and τ=10. If urban masks improve alignment before SC screening, that supports the *spatial-extent* argument; if τ changes the result within a mask, that supports the *cloud/observability* argument. Threshold comparisons use different date subsets, so neither change should be described as a randomized causal effect.

In [ ]:
intro_rows=[]
for scale,d in SCALES.items():
    for tau in TAUS:
        admitted=d.ntl.notna()&(d.SC_pct>=tau)&LOAD.notna()
        m=paired_metrics(d.ntl.where(admitted),LOAD,LOAD.index[LOAD.notna()])
        intro_rows.append({'scale':scale,'tau':tau,'direct_pct':100*admitted.sum()/LOAD.notna().sum(),**m})
INTRO=pd.DataFrame(intro_rows)
INTRO.to_csv(OUT/'reliability_conditions_direct_dates.csv',index=False)
fig=make_subplots(rows=1,cols=2,subplot_titles=['Daily support retained as SC τ rises','Raw direct alignment: extent then SC'],horizontal_spacing=.12)
for scale in MASK_ORDER:
    q=INTRO[INTRO.scale==scale]
    fig.add_trace(go.Scatter(x=q.tau,y=q.direct_pct,mode='lines+markers',name=scale,
                             line=dict(color=COLOR[scale],width=2)),row=1,col=1)
    z=q[q.tau.isin([0,10])]
    fig.add_trace(go.Scatter(x=z.tau,y=z.r,mode='lines+markers',name=scale,showlegend=False,
                             line=dict(color=COLOR[scale],width=2)),row=1,col=2)
fig.update_xaxes(title_text='SC threshold τ (%)')
fig.update_yaxes(title_text='Direct paired dates / load dates (%)',row=1,col=1)
fig.update_yaxes(title_text='Pearson r',row=1,col=2)
fig.update_layout(template='plotly_white',height=510,width=1330,
                  title='Why qualification has both spatial and observational components',
                  legend=dict(orientation='h',y=-.28),margin=dict(l=70,r=25,t=80,b=130),font=dict(color='#233B60'))
save_fig(fig,'fig_02_extent_and_sc_introduction')

## 3. Sensitivity cube: spatial scale × SC threshold × 1–30 calendar days

**Question.** What smoothing horizon is enough for a useful long-term signal, and how much support does it require? We compute **both** of the following lenses for every combination:

- **Direct-date lens (primary):** apply SC to daily direct NTL; smooth the admitted values over a centred *calendar* window; evaluate only dates with a directly admitted NTL observation and a load observation. `n%` is therefore normally constant across windows at fixed mask and τ. This makes the trade-off between $r$ and loss of directly observed dates explicit.
- **Window-supported lens (diagnostic):** require at least 50% of the calendar days in each window to contain an SC-admitted direct observation. An estimate can then exist on a date without a direct NTL observation. Its `n%` is **supported-estimate availability**, not directly observed date coverage. We report the median number of contributing direct days and show 25/50/75% support-rule sensitivity later.

The primary retrospective windows are centred, matching the spirit of Notebook 03's retrospective smoothing. A trailing-window check follows for prospective use. Both NTL and NGCP load are smoothed on the same calendar width; consequently rising $r$ describes agreement at a broader time scale, not better daily outage detection. `w=1` is unsmoothed direct NTL. This calendar-day sweep is **not** a reproduction of the original 30-retained-observation method.

In [ ]:
def smooth_pair(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
    admitted=d.ntl.notna()&(d.Valid_px>0)&(d.SC_pct>=tau)
    x=d.ntl.where(admitted)
    count=x.notna().astype(int).rolling(w,center=center,min_periods=1).sum()
    needed=1 if lens=='direct' else max(1,math.ceil(support_fraction*w))
    xs=x.rolling(w,center=center,min_periods=needed).mean()
    ys=load.rolling(w,center=center,min_periods=needed).mean()
    if lens=='direct':
        xs=xs.where(admitted)
    # Every scored date must still have an actual NGCP load datum.
    xs=xs.where(load.notna())
    ys=ys.where(load.notna())
    return xs,ys,admitted,count

def configuration_metric(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
    x,y,admitted,count=smooth_pair(d,load,tau,w,lens,support_fraction,center)
    m=paired_metrics(x,y,load.index[load.notna()])
    paired=x.notna()&y.notna()
    m.update(direct_n=int((admitted&load.notna()).sum()),
             direct_pct=100*(admitted&load.notna()).sum()/load.notna().sum(),
             median_contributing_days=float(count[paired].median()) if paired.any() else np.nan,
             supported_without_target_direct=int((paired&~admitted).sum()))
    return m

# Synthetic checks are only about logic, not the RQ1 research data.
toy_i=pd.date_range('2020-01-01',periods=5)
toy_d=pd.DataFrame({'ntl':[1.,np.nan,np.nan,4.,5.],
                    'Valid_px':[10,0,0,10,10],
                    'SC_pct':[100,0,0,100,100]},index=toy_i)
toy_l=pd.Series([1.,2.,3.,4.,5.],index=toy_i)
a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'direct')
assert a.isna().iloc[1] and ad.sum()==3
a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'window',.5)
assert c.iloc[3]>=2 and a.iloc[3]==a.iloc[3]

rows=[]
for scale,d in SCALES.items():
    for tau in TAUS:
        for w in WINDOWS:
            for lens in ['direct','window']:
                rows.append({'scale':scale,'tau':tau,'window_days':w,'lens':lens,'centered':True,
                             **configuration_metric(d,LOAD,tau,w,lens,MIN_WINDOW_FRACTION)})
MATRIX=pd.DataFrame(rows)
MATRIX.to_csv(OUT/'mask_sc_window_1_to_30_metric_cube.csv',index=False)
print(f'Saved {len(MATRIX):,} mask × SC × window × lens results. No result is selected yet.')

### Figure 3 — Window curves at declared SC thresholds

These curves address the reviewer's window question directly. Each panel fixes τ; every curve represents one spatial support. Read $r$ together with paired z-RMSE and direct-date `n%`. With paired z-scoring, $\mathrm{RMSE}^2=2(1-r)$, so RMSE is a useful familiar diagnostic but **not an independent score dimension**. A rise in $r$ with window length should be weighed against loss of temporal resolution even if direct `n%` stays flat. Full G1–G8 versions are exported; the concise main candidate shows full area plus G1–G3.

In [ ]:
def metric_curve_figure(df,metric,scales,taus,lens,name):
    f=make_subplots(rows=1,cols=len(taus),subplot_titles=[f'SC τ={t}%' for t in taus],
                    shared_yaxes=True,horizontal_spacing=.055)
    for col,tau in enumerate(taus,start=1):
        for scale in scales:
            q=df[(df.lens==lens)&(df.tau==tau)&(df.scale==scale)].sort_values('window_days')
            f.add_trace(go.Scatter(x=q.window_days,y=q[metric],mode='lines',name=scale,
                                   showlegend=(col==1),line=dict(color=COLOR[scale],width=2.2)),row=1,col=col)
    f.update_xaxes(title_text='Calendar window (days)',dtick=5)
    f.update_yaxes(title_text={'r':'Pearson r','rmse_z':'Paired z-score RMSE',
                               'n_pct':'Paired / load dates (%)',
                               'median_contributing_days':'Median direct days per estimate'}[metric],row=1,col=1)
    f.update_layout(template='plotly_white',width=1420,height=480,
                    title=name,legend=dict(orientation='h',y=-.25),
                    margin=dict(l=75,r=25,t=80,b=130),font=dict(color='#233B60'))
    return f

CORE_SCALES=MASK_ORDER[:4]
for metric,title,stem in [
    ('r','Alignment versus calendar window and SC','fig_03a_r_window_curves'),
    ('rmse_z','Paired standardized error versus calendar window and SC','fig_03b_rmse_window_curves'),
    ('n_pct','Directly paired dates versus calendar window and SC','fig_03c_direct_n_window_curves'),
]:
    save_fig(metric_curve_figure(MATRIX,metric,CORE_SCALES,[0,10,50],'direct',title),stem)

save_fig(metric_curve_figure(MATRIX,'r',MASK_ORDER,[0,10,50],'direct',
                             'Supplement: all spatial supports, direct-date r'),
         'fig_s_all_masks_r_window_curves')

### Figure 4 — The requested heatmap, with SC in the right dimension

SC itself is a **daily spatial fraction**, not a function of a 1–30-day temporal window. A heatmap with spatial scale on $y$ and window on $x$ must therefore color a *derived* quantity. We provide:

1. **Direct retention heatmap:** $y$ = spatial scale, $x$ = SC threshold, color = percentage of load dates with a directly admitted NTL observation. This is the clean view of the SC trade-off.
2. **Window-support heatmaps:** $y$ = spatial scale, $x$ = 1–30-day window, one panel per declared SC threshold, color = percentage of dates with a window estimate supported by at least half its calendar days. These dates are **not automatically directly observed**.
3. A matching $r$ heatmap on that same window-supported basis, so an apparent gain in availability is never presented without checking alignment.

The full cube CSV has $r$, RMSE, direct $n$, supported $n$, and median contributing days for every setting.

In [ ]:
direct_heat=INTRO.pivot(index='scale',columns='tau',values='direct_pct').reindex(MASK_ORDER)
fig=go.Figure(go.Heatmap(x=direct_heat.columns,y=direct_heat.index,z=direct_heat.values,
                         colorscale='Blues',zmin=0,zmax=100,colorbar=dict(title='Direct dates %'),
                         hovertemplate='Scale %{y}<br>SC τ=%{x}%<br>Direct dates=%{z:.1f}%<extra></extra>'))
fig.update_layout(template='plotly_white',width=1080,height=560,
                  title='Direct-date retention: spatial scale × SC threshold',
                  xaxis_title='SC threshold τ (%)',yaxis_title='Spatial scale',
                  margin=dict(l=190,r=35,t=75,b=70),font=dict(color='#233B60'))
save_fig(fig,'fig_04a_direct_retention_sc_heatmap')

def window_heatmap(metric,lens,taus,name,title):
    f=make_subplots(rows=1,cols=len(taus),subplot_titles=[f'SC τ={t}%' for t in taus],
                    shared_yaxes=True,horizontal_spacing=.035)
    for col,tau in enumerate(taus,start=1):
        q=MATRIX[(MATRIX.lens==lens)&(MATRIX.tau==tau)]
        p=q.pivot(index='scale',columns='window_days',values=metric).reindex(MASK_ORDER)
        f.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                               zmin=0 if metric=='n_pct' else None,
                               zmax=100 if metric=='n_pct' else 1,
                               showscale=(col==len(taus)),
                               colorbar=dict(title='Supported %' if metric=='n_pct' else 'Pearson r'),
                               hovertemplate='Scale %{y}<br>Window %{x} d<br>Value %{z:.2f}<extra></extra>'),row=1,col=col)
    f.update_xaxes(title_text='Calendar window (days)',dtick=5)
    f.update_yaxes(title_text='Spatial scale',row=1,col=1)
    f.update_layout(template='plotly_white',width=1450,height=570,title=title,
                    margin=dict(l=190,r=75,t=75,b=75),font=dict(color='#233B60'))
    save_fig(f,name)

window_heatmap('n_pct','window',[0,10,50],'fig_04b_supported_n_mask_window_heatmap',
               'Supported-estimate availability: spatial scale × window, faceted by SC')
window_heatmap('r','window',[0,10,50],'fig_04c_supported_r_mask_window_heatmap',
               'Alignment of supported window estimates: same grid and SC thresholds')

## 4. Choosing a window and score without hiding the trade-off

**First choose the decision rule, then inspect its result.** At fixed mask and τ, directly retained `n%` barely changes with window length, while smoothing often raises $r$. A simple score would therefore tend to choose the longest allowed window even if a shorter one performs almost as well. We use a transparent *resolution rule*: for each mask × τ, choose the **shortest** calendar window whose $r$ is within `Δr=0.02` of that pair's best 1–30-day $r$. We repeat this at 0.01 and 0.05. This is a practical tolerance for preserving temporal detail, not a significance test. Configuration candidates with <20% direct coverage or too few pairs are excluded.

**What is left to weight?** Paired standardized RMSE is algebraically tied to $r$ ($\mathrm{RMSE}^2=2(1-r)$). The independent priorities are therefore **alignment** and **direct temporal representation**. A primary two-axis score is

$$S_\lambda = \lambda\frac{r+1}{2} + (1-\lambda)\frac{n_{\rm direct}}{n_{\rm load}}, \qquad 0\leq\lambda\leq1.$$

The fixed $(r+1)/2$ and 0–1 coverage scales do not change when candidates are added. We show the manuscript's historical 0.6/0.1/0.3 *r / inverse-RMSE / coverage* score separately on this **new** candidate grid. It is a comparator of priorities, **not** a reproduction of Table 4: Notebook 04 used a different threshold grid, outlier rule, and 30-retained-observation processing.

In [ ]:
def plateau_candidates(frame,delta,group_keys=['scale','tau']):
    candidates=[]
    source=frame[(frame.lens=='direct')&frame.r.notna()&
                 (frame.n_pct>=100*MIN_DIRECT_COVERAGE)&(frame.n_pairs>=30)].copy()
    for _,g in source.groupby(group_keys,sort=False):
        best=g.r.max()
        pick=g[g.r>=best-delta].sort_values('window_days').iloc[0]
        candidates.append(pick)
    return pd.DataFrame(candidates).reset_index(drop=True) if candidates else pd.DataFrame()

LOCAL_CANDIDATES=plateau_candidates(MATRIX,PLATEAU_DELTA_R)
LOCAL_CANDIDATES=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale!='Full RQ1 regional export'].copy()
if LOCAL_CANDIDATES.empty:
    raise ValueError('No locally eligible GHSL candidates: inspect n, SC and minimum coverage')
LOCAL_CANDIDATES.to_csv(OUT/'local_shortest_near_plateau_candidates.csv',index=False)

def primary_score(frame,lam):
    q=frame.copy()
    q['score']=lam*(q.r+1)/2+(1-lam)*q.n_pct/100
    return q.sort_values(['score','n_pct','window_days','scale','tau'],
                         ascending=[False,False,True,True,True]).reset_index(drop=True)

LOCAL_WEIGHTS=[]
for lam in np.round(np.arange(0,1.0001,.05),2):
    winner=primary_score(LOCAL_CANDIDATES,lam).iloc[0]
    LOCAL_WEIGHTS.append({'lambda_alignment':float(lam),'selected_scale':winner.scale,
                          'selected_tau':int(winner.tau),'selected_window_days':int(winner.window_days),
                          'r':winner.r,'rmse_z':winner.rmse_z,'n_pairs':int(winner.n_pairs),
                          'n_pct':winner.n_pct,'score':winner.score})
LOCAL_WEIGHTS=pd.DataFrame(LOCAL_WEIGHTS)
LOCAL_WEIGHTS.to_csv(OUT/'local_weight_sweep_selected_configurations.csv',index=False)

def minmax(s):
    span=s.max()-s.min()
    return pd.Series(.5,index=s.index) if not np.isfinite(span) or span==0 else (s-s.min())/span
historic=LOCAL_CANDIDATES.copy()
historic['historical_score_on_new_grid']=(.6*minmax(historic.r)
                                          +.1*minmax(-historic.rmse_z)
                                          +.3*minmax(historic.n_pct))
historic=historic.sort_values('historical_score_on_new_grid',ascending=False)
historic.to_csv(OUT/'historical_weights_on_new_grid_NOT_Table4.csv',index=False)
print('Historical 0.6/0.1/0.3 applied to new window grid:',
      historic.iloc[0][['scale','tau','window_days','r','n_pct']].to_dict())
print('Maximum paired-z identity discrepancy:',
      float(np.nanmax(abs(LOCAL_CANDIDATES.rmse_z**2-2*(1-LOCAL_CANDIDATES.r)))))

plateau_rows=[]
for delta in [.01,.02,.05]:
    q=plateau_candidates(MATRIX,delta)
    q=q[q.scale.isin(['G1','G2','G3'])]
    for lam in [.3,.5,.7]:
        if len(q):
            x=primary_score(q,lam).iloc[0]
            plateau_rows.append({'delta_r':delta,'lambda_alignment':lam,'scale':x.scale,
                                 'tau':x.tau,'window_days':x.window_days,'r':x.r,'n_pct':x.n_pct})
PLATEAU_CHECK=pd.DataFrame(plateau_rows)
PLATEAU_CHECK.to_csv(OUT/'window_plateau_tolerance_sensitivity.csv',index=False)

### Figure 5 — Pareto choices and weight sensitivity

The scatter places every shortest-near-plateau mask × τ setting on the **alignment versus directly paired coverage** plane; point color is selected window length. The next chart shows the configuration picked as the alignment weight moves from 0 to 1. These are stated preferences, not an optimization proof. A configuration that wins at one weight need not generalize to a region where NGCP is unavailable.

In [ ]:
fig=go.Figure()
for scale in [v[0] for v in GROUPS.values()]:
    q=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale==scale]
    if q.empty: continue
    fig.add_trace(go.Scatter(x=q.n_pct,y=q.r,mode='markers',name=scale,
                             marker=dict(size=9,color=COLOR[scale],opacity=.72),
                             customdata=np.stack([q.tau,q.window_days],axis=1),
                             hovertemplate=f'{scale}<br>SC τ=%{{customdata[0]}}%<br>Window=%{{customdata[1]}} d<br>Direct N=%{{x:.1f}}%<br>r=%{{y:.3f}}<extra></extra>'))
fig.update_layout(template='plotly_white',width=1030,height=570,
                  title='Candidate frontier after the shortest-near-plateau window rule',
                  xaxis_title='Directly paired load dates (%)',yaxis_title='Pearson r',
                  legend=dict(orientation='h',y=-.21),margin=dict(l=70,r=25,t=75,b=115),font=dict(color='#233B60'))
save_fig(fig,'fig_05a_alignment_coverage_candidate_frontier')

fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.10,
                  subplot_titles=['Selected SC threshold and mask','Alignment and direct-date coverage of selection'])
for scale in [v[0] for v in GROUPS.values()]:
    q=LOCAL_WEIGHTS[LOCAL_WEIGHTS.selected_scale==scale]
    if q.empty: continue
    fig.add_trace(go.Scatter(x=q.lambda_alignment,y=q.selected_tau,mode='markers',name=scale,
                             marker=dict(size=10,color=COLOR[scale]),
                             text=[f'{w} d' for w in q.selected_window_days],
                             hovertemplate='Alignment weight %{x:.2f}<br>SC τ=%{y}%<br>Window %{text}<extra>'+scale+'</extra>'),row=1,col=1)
fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.r,mode='lines+markers',
                         name='Selected r',line=dict(color='#4E79A7',width=2)),row=2,col=1)
fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.n_pct/100,
                         mode='lines+markers',name='Selected direct fraction',
                         line=dict(color='#375623',width=2)),row=2,col=1)
fig.update_xaxes(title_text='Declared alignment priority λ',row=2,col=1)
fig.update_yaxes(title_text='SC threshold τ (%)',row=1,col=1)
fig.update_yaxes(title_text='r / fraction',row=2,col=1)
fig.update_layout(template='plotly_white',width=1110,height=660,
                  title='What changes when the application values alignment more than temporal coverage?',
                  legend=dict(orientation='h',y=-.13),margin=dict(l=75,r=25,t=80,b=110),font=dict(color='#233B60'))
save_fig(fig,'fig_05b_local_weight_sensitivity')

## 5. Does the *selection rule* transfer when NGCP is unavailable locally?

A score containing $r$ or RMSE **cannot be recomputed in a target region without NGCP**. The testable alternative is to learn one fixed mask × SC × window *policy* on other islands, apply it unchanged to the held-out island, and reveal that island's NGCP only for evaluation. This is a five-fold, leave-one-sub-grid-out diagnostic using the RQ1 Notebook 04 CSVs and workbook. It tests practical transfer more honestly than optimizing weights and reporting the same islands' in-sample scores. Five islands remain exploratory evidence, not a universal guarantee.

For each held-out island and each declared weight, the code: (i) computes the shortest near-plateau window for every mask × τ from the four **training** islands only; (ii) ranks those shared candidates by the median training-island score, requiring ≥20% direct paired dates on every training island; (iii) applies the winner to the held-out island without using its load during selection; and (iv) compares held-out $r$ and `n%` with a **G8 τ=0, one-day** comparator and a clearly labelled *hindsight* local oracle. The G8 comparator still has the common 95th-percentile cap; it is **not** the whole geographic footprint, because RQ1 Notebook 04 has no full-footprint CSV for every island.

In [ ]:
SHEET_MAP={
    'Panay':'PANAY HOURLY LOAD 2013-2024',
    'Negros':'NEGROS HOURLY LOAD 2013-2024',
    'Samar-Leyte':'LEY-SAM HOURLY LOAD 2013-2024',
    'Cebu':'CEBU HOURLY LOAD 2013-2024',
    'Bohol':'BOHOL LOAD 2013-2024',
}
REGION_NAMES={'SamarLeyte':'Samar-Leyte','BoholProvince':'Bohol',
              'CebuProvince':'Cebu','NegrosIsland':'Negros','PanayIsland':'Panay'}

def cross_region_inputs(folder,workbook):
    if not folder.is_dir() or not workbook.is_file():
        return {},{},f'Missing RQ1 Notebook 04 data: {folder} or {workbook}'
    files=sorted(folder.glob('*_DNBBRDF_class*_stats.csv'))
    if not files:
        return {},{},f'No Notebook 04 mask CSVs found in {folder}'
    region_data={}
    for path in files:
        parts=path.name.replace('_stats.csv','').split('_DNBBRDF_class')
        if len(parts)!=2 or parts[0] not in REGION_NAMES:
            continue
        region=REGION_NAMES[parts[0]]
        group='30' if parts[1]=='30-only' else parts[1]
        if group not in GROUPS: continue
        region_data[(region,GROUPS[group][0])]=prep_direct(read_daily_csv(path,'Mean_by_sum'))[0]
    loads={}
    for region,sheet in SHEET_MAP.items():
        f=pd.read_excel(workbook,sheet_name=sheet,header=1)
        f.columns=[str(x).strip() for x in f.columns]
        date_col=f.columns[0]
        dates=pd.to_datetime(f[date_col],errors='coerce').dt.normalize()
        hour_col=next((c for c in ['1','01','1.0','Hour 1','HOUR 1','1AM','1am'] if c in f.columns),None)
        if hour_col is None:
            raise ValueError(f'{sheet}: NGCP Hour 1 column not found; inspect workbook rather than using another hour')
        s=pd.Series(pd.to_numeric(f[hour_col],errors='coerce').to_numpy(),index=dates).dropna()
        s=s[~s.index.isna()]
        if s.index.duplicated().any():
            raise ValueError(f'{sheet}: duplicate dates')
        loads[region]=s.reindex(CALENDAR)
    missing={(region,g) for region in SHEET_MAP for g in [v[0] for v in GROUPS.values()]}-set(region_data)
    if missing:
        return {},{},f'Missing region × GHSL mask inputs: {sorted(missing)}'
    return region_data,loads,''

CROSS_DATA,CROSS_LOAD,CROSS_STATUS=cross_region_inputs(CROSS_DIR,NGCP_XLSX)
print(CROSS_STATUS or f'Loaded {len(CROSS_DATA)} region × mask direct inputs and {len(CROSS_LOAD)} NGCP regions.')

In [ ]:
CROSS_MATRIX=pd.DataFrame()
if not CROSS_STATUS:
    cross_rows=[]
    for (region,scale),d in CROSS_DATA.items():
        load=CROSS_LOAD[region]
        for tau in TRANSFER_TAUS:
            for w in WINDOWS:
                cross_rows.append({'region':region,'scale':scale,'tau':tau,'window_days':w,
                                   'lens':'direct',**configuration_metric(d,load,tau,w,'direct')})
    CROSS_MATRIX=pd.DataFrame(cross_rows)
    CROSS_MATRIX.to_csv(OUT/'cross_island_mask_sc_window_metric_cube.csv',index=False)
    print(f'Saved {len(CROSS_MATRIX):,} direct-date cross-island candidates.')
else:
    (OUT/'cross_island_status.txt').write_text(CROSS_STATUS+'\n')

# Optional supplementary comparison using only the direct summaries available for all islands.
# A 4-day rolling mean of regional CSV values is NOT the pixel-first Roman-inspired block composite.
if not CROSS_MATRIX.empty:
    comparison=CROSS_MATRIX[(CROSS_MATRIX.tau==0)&(CROSS_MATRIX.window_days.isin([1,4]))].copy()
    comparison['method']=np.where(comparison.window_days==1,'Direct daily regional summary',
                                  '4-day calendar mean of regional summaries')
    comparison[['region','scale','method','r','rmse_z','n_pairs','n_pct']].to_csv(
        OUT/'supplement_cross_island_direct_summary_comparison_NO_gapfilled.csv',index=False)

In [ ]:
def transfer_fold(matrix,holdout,lam,delta,region_data=None,region_load=None):
    train=matrix[matrix.region!=holdout].copy()
    train=train[train.r.notna()&(train.n_pairs>=30)]
    target=matrix[matrix.region==holdout].copy()
    regions=set(train.region.unique())
    if len(regions)!=4:
        raise ValueError(f'{holdout}: expected four training islands')
    # A candidate must be present and sufficiently observed in every training region.
    agg=(train.groupby(['scale','tau','window_days'],as_index=False)
         .agg(median_r=('r','median'),median_pct=('n_pct','median'),
              min_pct=('n_pct','min'),regions=('region','nunique')))
    agg=agg[(agg.regions==4)&(agg.min_pct>=100*MIN_DIRECT_COVERAGE)]
    choices=[]
    for _,g in agg.groupby(['scale','tau'],sort=False):
        best=g.median_r.max()
        choices.append(g[g.median_r>=best-delta].sort_values('window_days').iloc[0])
    if not choices:
        raise ValueError(f'{holdout}: no transfer candidate survives training coverage rule')
    choices=pd.DataFrame(choices)
    choices['training_score']=lam*(choices.median_r+1)/2+(1-lam)*choices.median_pct/100
    selected=choices.sort_values(['training_score','median_pct','window_days','scale','tau'],
                                 ascending=[False,False,True,True,True]).iloc[0]
    key=(target.scale==selected.scale)&(target.tau==selected.tau)&(target.window_days==selected.window_days)
    if key.sum()!=1:
        raise ValueError(f'{holdout}: selected configuration missing from held-out metric matrix')
    evaluated=target[key].iloc[0]
    baseline=target[(target.scale=='G8')&(target.tau==0)&(target.window_days==1)].iloc[0]
    valid_heldout=bool(np.isfinite(evaluated.r) and evaluated.n_pairs>=30)
    # Hindsight comparison uses held-out load and the SAME near-plateau rule.
    # It is reported only as an optimistic benchmark, never used for selection.
    hindsight=target[target.r.notna()&(target.n_pairs>=30)&
                     (target.n_pct>=100*MIN_DIRECT_COVERAGE)].copy()
    local_choices=[]
    for _,g in hindsight.groupby(['scale','tau'],sort=False):
        best=g.r.max()
        local_choices.append(g[g.r>=best-delta].sort_values('window_days').iloc[0])
    if not local_choices:
        raise ValueError(f'{holdout}: no eligible hindsight comparison')
    local_choices=pd.DataFrame(local_choices)
    local_choices['oracle_score']=lam*(local_choices.r+1)/2+(1-lam)*local_choices.n_pct/100
    oracle=local_choices.sort_values(['oracle_score','n_pct'],ascending=False).iloc[0]

    # Isolate spatial/SC choices: compare selected NTL with G8 at the SAME
    # window, against the SAME smoothed load, and on the SAME calendar dates.
    selected_common_r, g8_common_r, common_n=np.nan,np.nan,0
    if region_data is not None and region_load is not None:
        held_load=region_load[holdout]
        sx,sy,_,_=smooth_pair(region_data[(holdout,selected.scale)],held_load,
                              int(selected.tau),int(selected.window_days),'direct')
        bx,_,_,_=smooth_pair(region_data[(holdout,'G8')],held_load,0,
                             int(selected.window_days),'direct')
        common=held_load.index[sx.notna()&bx.notna()&sy.notna()]
        sm=paired_metrics(sx,sy,common)
        bm=paired_metrics(bx,sy,common)
        selected_common_r,g8_common_r,common_n=sm['r'],bm['r'],sm['n_pairs']
    return {'held_out':holdout,'lambda_alignment':lam,'delta_r':delta,
            'selected_mask':selected.scale,'selected_tau':int(selected.tau),
            'selected_window_days':int(selected.window_days),
            'training_score':float(selected.training_score),
            'held_out_r':evaluated.r,'held_out_rmse_z':evaluated.rmse_z,
            'held_out_n':int(evaluated.n_pairs),'held_out_n_pct':evaluated.n_pct,
            'valid_heldout':valid_heldout,
            'selected_common_r':selected_common_r,'g8_same_window_common_r':g8_common_r,
            'common_pair_n':common_n,
            'baseline_G8_raw_r':baseline.r,'baseline_G8_raw_n_pct':baseline.n_pct,
            'hindsight_oracle_r':oracle.r,'hindsight_oracle_n_pct':oracle.n_pct,
            'hindsight_oracle_mask':oracle.scale,'hindsight_oracle_tau':int(oracle.tau),
            'hindsight_oracle_window_days':int(oracle.window_days)}

TRANSFER=pd.DataFrame()
if not CROSS_MATRIX.empty:
    transfer_rows=[]
    for holdout in SHEET_MAP:
        for lam in [.3,.5,.7]:
            for delta in [.01,.02,.05]:
                transfer_rows.append(transfer_fold(CROSS_MATRIX,holdout,lam,delta,CROSS_DATA,CROSS_LOAD))
    TRANSFER=pd.DataFrame(transfer_rows)
    TRANSFER.to_csv(OUT/'leave_one_island_out_weight_and_window_transfer.csv',index=False)
    print(f'Saved {len(TRANSFER)} held-out evaluations (five islands × priorities × window tolerances).')
    if (~TRANSFER.valid_heldout).any():
        print('ATTENTION: Some selected held-out configurations have fewer than 30 pairs or undefined r.')
        display(TRANSFER.loc[~TRANSFER.valid_heldout,
                         ['held_out','lambda_alignment','delta_r','selected_mask','selected_tau',
                          'selected_window_days','held_out_n']])

### Figure 6 — Generalizability is a held-out question

This chart displays the predeclared `λ=0.5, Δr=0.02` policy for each held-out island. It shows held-out $r$ **and** direct-date fraction, alongside the capped G8 τ=0, one-day comparator and the held-out *hindsight* best score (using the same window rule). That whole-pipeline G8 comparison changes smoothing as well as mask. The exported table adds a **same-window, same-date G8 comparison** to isolate the mask/SC contribution. Figure 6B then checks the held-out average across all declared weights and window tolerances; cells are hidden unless all five folds have valid $r$. If the transferred choices remain useful across grids, that supports transport of the **selection logic**; if they vary or underperform, report that limitation and consider a smaller family of application-specific policies. It does **not** justify calculating an $r$-weighted score where load is unavailable.

In [ ]:
if not TRANSFER.empty:
    q=TRANSFER[(TRANSFER.lambda_alignment==.5)&(TRANSFER.delta_r==.02)].copy()
    q=q[q.valid_heldout].copy()
    if q.empty:
        raise ValueError('No valid held-out fold for the declared λ=0.5, Δr=0.02 display')
    order=list(SHEET_MAP)
    q['held_out']=pd.Categorical(q.held_out,categories=order,ordered=True)
    q=q.sort_values('held_out')
    fig=make_subplots(rows=1,cols=2,subplot_titles=['Held-out Pearson r','Held-out directly paired load dates (%)'],
                      horizontal_spacing=.11)
    for col,triples in [(1,[('G8 τ=0, 1 d','baseline_G8_raw_r','#A8CE74'),
                           ('Transferred rule','held_out_r','#4E79A7'),
                           ('Hindsight local best','hindsight_oracle_r','#D65A4A')]),
                        (2,[('G8 τ=0, 1 d','baseline_G8_raw_n_pct','#A8CE74'),
                            ('Transferred rule','held_out_n_pct','#4E79A7'),
                            ('Hindsight local best','hindsight_oracle_n_pct','#D65A4A')])]:
        for name,key,color in triples:
            fig.add_trace(go.Scatter(x=q.held_out.astype(str),y=q[key],mode='markers+lines',
                                     name=name,showlegend=(col==1),
                                     marker=dict(size=10,color=color),line=dict(color=color,width=1.5)),row=1,col=col)
    fig.update_yaxes(title_text='Pearson r',row=1,col=1)
    fig.update_yaxes(title_text='Directly paired dates (%)',row=1,col=2)
    fig.update_xaxes(title_text='Held-out island grid',tickangle=-20)
    fig.update_layout(template='plotly_white',width=1250,height=510,
                      title='Cross-island transfer of a fixed reliability policy (λ=0.5; Δr=0.02)',
                      legend=dict(orientation='h',y=-.22),margin=dict(l=70,r=30,t=80,b=120),font=dict(color='#233B60'))
    save_fig(fig,'fig_06_held_out_rule_transfer')
    display(q[['held_out','selected_mask','selected_tau','selected_window_days',
               'held_out_r','held_out_n','held_out_n_pct','common_pair_n',
               'selected_common_r','g8_same_window_common_r']].round(3))

    summary=(TRANSFER[TRANSFER.valid_heldout]
             .groupby(['lambda_alignment','delta_r'],as_index=False)
             .agg(mean_heldout_r=('held_out_r','mean'),
                  mean_heldout_n_pct=('held_out_n_pct','mean'),
                  valid_folds=('held_out','nunique')))
    summary.loc[summary.valid_folds<5,['mean_heldout_r','mean_heldout_n_pct']]=np.nan
    summary.to_csv(OUT/'transfer_priority_sensitivity_summary.csv',index=False)
    fig=make_subplots(rows=1,cols=2,subplot_titles=['Mean held-out r','Mean direct-date retention (%)'],
                      horizontal_spacing=.15)
    for col,field,zmin,zmax,title in [(1,'mean_heldout_r',-1,1,'Pearson r'),
                                     (2,'mean_heldout_n_pct',0,100,'Direct N %')]:
        p=summary.pivot(index='delta_r',columns='lambda_alignment',values=field)
        fig.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                                 zmin=zmin,zmax=zmax,colorbar=dict(title=title),
                                 hovertemplate='λ=%{x}<br>Δr=%{y}<br>Mean=%{z:.3f}<extra></extra>'),row=1,col=col)
    fig.update_xaxes(title_text='Alignment priority λ')
    fig.update_yaxes(title_text='Near-plateau tolerance Δr',row=1,col=1)
    fig.update_layout(template='plotly_white',width=1150,height=470,
                      title='Held-out sensitivity across declared priorities (only complete five-fold cells)',
                      margin=dict(l=80,r=80,t=80,b=70),font=dict(color='#233B60'))
    save_fig(fig,'fig_06b_held_out_priority_sensitivity')
else:
    print('Figure 6 awaits the RQ1 Notebook 04 cross-island input folder and NGCP workbook; no transfer claim is made.')

## Where the figures belong

**Main paper:** Figure 1A–B (2012–2024 baseline from the paired Region VIII CSV); Figure 2 (why SC and GHSL are distinct); Figure 3 (1–30-day $r$ curves with direct $n$ at τ=0/10/50); a compact Figure 5 (declared alignment–coverage priorities). **Supplement:** Haiyan and Rai zooms; RMSE curves; all-mask heatmaps; complete sensitivity matrices; held-out transfer Figure 6 unless the transfer result becomes a central claim.

The 2012–2024 window and the geographic footprint must be stated in every caption. Figure 4's direct-retention heatmap varies with mask and SC threshold; supported-window coverage varies with smoothing width but is **not** direct daily observation. The manuscript's original 30 retained observations and this notebook's 30 calendar days are distinct. No results have been pre-interpreted here.